In [3]:
import os
import glob
import fitz  # PyMuPDF 库

def merge_invoices_with_paste_sheet():
    # 1. 准备发票文件 (现在去“待报销的发票”文件夹里找)
    invoice_folder = "待报销的发票"
    # 使用 os.path.join 拼凑路径，自动搜索该文件夹下的所有 pdf
    pdf_files = glob.glob(os.path.join(invoice_folder, "*.pdf"))
    
    # 设定输出文件名 (保存在主目录，不混进发票文件夹)
    output_filename = "最终_合并后的发票_带粘贴单.pdf"

    if not pdf_files:
        print(f"在【{invoice_folder}】文件夹里没有找到发票 PDF 文件哦！请检查一下。")
        return

    # 2. 准备粘贴单文件
    paste_sheet_path = os.path.join("粘贴单", "原始票据（_影像化）报销粘贴单.pdf")
    if not os.path.exists(paste_sheet_path):
        print(f"找不到粘贴单！请确保存在这个文件：{paste_sheet_path}")
        return

    print(f"在【{invoice_folder}】里检测到 {len(pdf_files)} 张发票，正在交替排版...")

    # 创建一个新的空白 PDF 文档用于保存最终结果
    doc_out = fitz.open()
    # 打开预先准备好的粘贴单 PDF
    doc_paste = fitz.open(paste_sheet_path)
    
    # 定义 A4 竖向的尺寸和上下两块“画框”
    a4_width = 595.0
    a4_height = 842.0
    rect_top = fitz.Rect(0, 0, a4_width, a4_height / 2)       # 上半张纸
    rect_bottom = fitz.Rect(0, a4_height / 2, a4_width, a4_height) # 下半张纸

    # 3. 开始循环处理：每次取两张发票
    for i in range(0, len(pdf_files), 2):
        # 新建一页竖向 A4 空白纸，拼版发票
        page = doc_out.new_page(width=a4_width, height=a4_height)
        
        # 处理第一张发票 (上半部分)
        doc_1 = fitz.open(pdf_files[i])
        page.show_pdf_page(rect_top, doc_1, 0)
        doc_1.close()
        
        # 处理第二张发票 (下半部分，如果有的话)
        if i + 1 < len(pdf_files):
            doc_2 = fitz.open(pdf_files[i+1])
            page.show_pdf_page(rect_bottom, doc_2, 0)
            doc_2.close()

        # 在发票拼版页后面，立刻插入一页粘贴单
        doc_out.insert_pdf(doc_paste, from_page=0, to_page=0)

    # 4. 收尾并导出最终的 PDF
    doc_paste.close()
    doc_out.save(output_filename)
    doc_out.close()
    
    print(f"大功告成！已生成交替排版的【{output_filename}】，保存在代码所在的文件夹中。")

if __name__ == "__main__":
    merge_invoices_with_paste_sheet()

在【待报销的发票】里检测到 1 张发票，正在交替排版...
大功告成！已生成交替排版的【最终_合并后的发票_带粘贴单.pdf】，保存在代码所在的文件夹中。
